# Introduction to Transformers and Self-Attention
This notebook will guide you through the fundamental building blocks of modern Large Language Models (LLMs): The **Transformer Architecture** and the **Self-Attention Mechanism**. 
Transformers are the highest-priority topic in modern Deep Learning today.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense
print('TensorFlow version:', tf.__version__)

## 1. The Self-Attention Mechanism
Unlike RNNs or LSTMs that process data sequentially, self-attention allows the model to look at the entire sequence at once and weigh the importance of different words in relation to a specific word.

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    '''
    q: Queries (What we are looking for)
    k: Keys (What I have)
    v: Values (The actual content)
    '''
    # 1. Matmul Q and K to get the raw attention scores
    matmul_qk = tf.matmul(q, k, transpose_b=True)
    
    # 2. Scale by the square root of the key dimension depth
    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
    
    # Optional: Apply masking (used in Decoders so they don't "look ahead")
    if mask is not None:
        scaled_attention_logits += (mask * -1e9)
        
    # 3. Apply Softmax to get probabilities (attention weights)
    attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
    
    # 4. Multiply by Values to get the final output
    output = tf.matmul(attention_weights, v)
    
    return output, attention_weights

In [ ]:
# Let's test the attention mechanism with some dummy data
seq_len = 3
depth = 4
np.random.seed(42)

# Create random Queries, Keys, and Values representing 3 words with embeddings of size 4
q = np.random.rand(1, seq_len, depth).astype(np.float32)
k = np.random.rand(1, seq_len, depth).astype(np.float32)
v = np.random.rand(1, seq_len, depth).astype(np.float32)

output, weights = scaled_dot_product_attention(q, k, v)

print('Attention Weights (How much each word pays attention to others):')
print(weights.numpy())
print('\nOutput (Contextualized embeddings):')
print(output.numpy())

## 2. Multi-Head Attention
Instead of doing attention once, Transformers do it multiple times in parallel ("heads"). This allows the model to focus on different aspects of the text simultaneously (e.g., one head focuses on grammar, another on subject-verb relationships).

In [ ]:
class MultiHeadAttention(Layer):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        
        assert d_model % self.num_heads == 0
        self.depth = d_model // self.num_heads
        
        self.wq = Dense(d_model)
        self.wk = Dense(d_model)
        self.wv = Dense(d_model)
        
        self.dense = Dense(d_model)
        
    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])
    
    def call(self, v, k, q, mask):
        batch_size = tf.shape(q)[0]
        
        # Pass through dense layers
        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)
        
        # Split into multiple heads
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        
        # Apply scaled dot product attention
        scaled_attention, attention_weights = scaled_dot_product_attention(q, k, v, mask)
        
        # Re-concatenate the heads
        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.d_model))
        
        # Final dense layer
        output = self.dense(concat_attention)
        return output, attention_weights

In [ ]:
# Test Multi-Head Attention
mha = MultiHeadAttention(d_model=16, num_heads=4)
y = tf.random.uniform((1, 3, 16))  # (batch_size, seq_len, d_model)
out, attn_weights = mha(y, k=y, q=y, mask=None)
print("Multi-Head Attention Output Shape:", out.shape)